# Cluster TFs by motif co-occurrence — AbbasTFScreen

Groups TFs whose HOCOMOCO motifs co-occur across the cell-state peak set, so the
TF-activity regression (`05_TFActivityRegression`) is not destabilised by collinear
motifs. Mirrors EpigeneticInhibitors `07_ClusterTFs.ipynb` / PERCISTRA `R/tf_clustering.R`.

**Order:** run after `02_GenerateMotifMatrix_TFScreen.r`, before the regression.
**Tune `JACCARD_THRESHOLD` below, re-run, and inspect the graph + heatmap.**

Outputs (to the `CSVFiles` dir):
- `TFGroups_hocomoco.csv` — TF_name, TF_group
- `MotifMatrixTFGroups_hocomoco.csv` — peaks x (TF groups + ungrouped TFs); the design matrix for the regression

In [ ]:
suppressPackageStartupMessages({
    library(igraph)
    library(reshape2)
    library(pheatmap)
})
`%ni%` <- Negate(`%in%`)

In [ ]:
# ---- paths / tunable threshold ----
CSV   <- "/data/AbbasTFScreen/TFScreenATAC_cellstate/DiffPeaks/CSVFiles"
MOTIF <- file.path(CSV, "MotifMatrix_hocomoco.csv")   # peaks x TF (0/1), from stage 02

JACCARD_THRESHOLD <- 0.5   # TF-TF edge kept if Jaccard > this. Tune and re-run.

# outputs are saved in the project (not /data):
OUT <- "/home/eraslab1/Projects/AbbasTFScreen/CSV_Files"
dir.create(OUT, showWarnings = FALSE, recursive = TRUE)


In [ ]:
# ---- load the binary peaks x TF motif matrix ----
myMotifMatrix <- read.csv(MOTIF, row.names = 1, check.names = FALSE)
cat("motif matrix:", nrow(myMotifMatrix), "peaks x", ncol(myMotifMatrix), "TFs\n")
myMotifMatrix

In [ ]:
# ---- TF x TF Jaccard co-occurrence ----
# fast crossprod form; identical to the pairwise jaccard(a,b)=|a&b|/|a|b| loop
tf_jaccard_matrix <- function(m) {
    m <- as.matrix(m); storage.mode(m) <- "double"
    inter <- crossprod(m)                       # |A_i & A_j|
    csum  <- colSums(m)
    union <- outer(csum, csum, "+") - inter     # |A_i | A_j|
    as.data.frame(ifelse(union == 0, 0, inter / union))
}
res <- tf_jaccard_matrix(myMotifMatrix)
offdiag <- res[upper.tri(as.matrix(res))]
cat("off-diagonal Jaccard: min", round(min(offdiag),3),
    " max", round(max(offdiag),3),
    " (pairs > threshold:", sum(offdiag > JACCARD_THRESHOLD), ")\n")

### Choose a threshold
Histogram of nonzero off-diagonal Jaccard values; the red line is the current `JACCARD_THRESHOLD`.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)
hist(offdiag[offdiag > 0], breaks = 60,
     main = "off-diagonal TF-TF Jaccard (> 0)", xlab = "Jaccard")
abline(v = JACCARD_THRESHOLD, col = "red", lwd = 2)

### Similarity graph
TFs connected when Jaccard > threshold (Fruchterman-Reingold layout). Connected components ~ the TF groups.

In [ ]:
resX <- res; resX$X <- rownames(resX)
myM <- melt(resX, id.vars = "X")
myM <- myM[myM$X != myM$variable, ]
myMTmp <- myM[myM$value > JACCARD_THRESHOLD, ]

myGraph <- graph_from_data_frame(myMTmp, directed = FALSE)
cat("graph:", vcount(myGraph), "TFs,", ecount(myGraph), "edges\n")

options(repr.plot.width = 15, repr.plot.height = 15)
set.seed(1); coords <- layout_with_fr(myGraph)
plot(myGraph, layout = coords, vertex.size = 3, vertex.label.cex = 0.6)

### Community detection (`cluster_leading_eigen`)

In [ ]:
c1 <- cluster_leading_eigen(myGraph)
k  <- membership(c1); k <- k[order(k)]
myTFGroups <- data.frame(TF_name = names(k), TF_group = as.integer(k),
                         row.names = NULL, stringsAsFactors = FALSE)
cat(length(unique(myTFGroups$TF_group)), "groups covering",
    nrow(myTFGroups), "TFs\n")
table(myTFGroups$TF_group)

### Inspect the grouping — annotated Jaccard heatmap
TFs ordered by group; rows/cols annotated by `TF_group` (values clipped at the threshold for contrast). Coherent blocks along the diagonal = good groups.

In [ ]:
annot <- myTFGroups
kk <- as.data.frame(table(annot$TF_group)); kk <- kk[order(-kk$Freq), ]
annot$TF_group <- factor(paste0("TFGroup_", annot$TF_group),
                         levels = paste0("TFGroup_", kk$Var1))
annot <- annot[order(annot$TF_group), ]
rownames(annot) <- annot$TF_name
cov <- res[annot$TF_name, annot$TF_name]
cov[cov < JACCARD_THRESHOLD] <- JACCARD_THRESHOLD
annot$TF_name <- NULL

options(repr.plot.width = 12, repr.plot.height = 12)
pheatmap(cov,
         cluster_rows = FALSE, cluster_cols = FALSE,
         annotation_row = annot, annotation_col = annot,
         fontsize_row = 4, fontsize_col = 4,
         color = colorRampPalette(c("blue", "white", "red"))(100),
         main = paste0("TF-TF Jaccard, grouped (threshold = ", JACCARD_THRESHOLD, ")"))

### Optional: where do the screened / perturbation TFs land?
HOCOMOCO uses UniProt-style names for some (NR3C1->GCR, NEUROD1->NDF1, NEUROG1->NGN1, TWIST1->TWST1).

In [ ]:
screened <- c("ASCL1","KLF14","NDF1","NGN1","GCR","SIM1","TWST1","VSX1","NEUROD1",
               "NEUROG1","NR3C1","TWIST1")
myTFGroups[myTFGroups$TF_name %in% screened, ]

### Write TF groups + collapse the motif matrix by group
Ungrouped TFs are kept as their own columns; grouped TFs become one `TFGroup_<n>` column = per-peak max over members.

In [ ]:
# complete table: grouped TFs relabelled TFGroup_<n>, ungrouped appended as "NA"
TFGroups <- myTFGroups
TFGroups$TF_group <- paste0("TFGroup_", TFGroups$TF_group)
TFGroups$TF_group <- factor(TFGroups$TF_group, levels = unique(TFGroups$TF_group))
TFGroups <- TFGroups[order(TFGroups$TF_group, TFGroups$TF_name), ]
otherTFs <- colnames(myMotifMatrix)[colnames(myMotifMatrix) %ni% TFGroups$TF_name]
s <- data.frame(TF_name = sort(otherTFs), TF_group = "NA", stringsAsFactors = FALSE)
TFGroups <- rbind(data.frame(TF_name = TFGroups$TF_name,
                             TF_group = as.character(TFGroups$TF_group),
                             stringsAsFactors = FALSE), s)
write.csv(TFGroups, file.path(OUT, "TFGroups_hocomoco.csv"), row.names = FALSE)
cat(nrow(TFGroups), "TFs ->", length(unique(TFGroups$TF_group)), "labels\n")

In [ ]:
# collapse: one column per group (max over members), ungrouped TFs untouched
grouped <- myMotifMatrix
for (g in unique(myTFGroups$TF_group)) {
    members <- myTFGroups[myTFGroups$TF_group == g, "TF_name"]
    members <- intersect(members, colnames(grouped))
    if (length(members) == 0) next
    grouped[[paste0("TFGroup_", g)]] <- apply(grouped[, members, drop = FALSE], 1, max)
    grouped[, members] <- NULL
}
cat("grouped motif matrix:", nrow(grouped), "peaks x", ncol(grouped),
    "columns (TF groups + ungrouped TFs)\n")
write.csv(grouped, file.path(OUT, "MotifMatrixTFGroups_hocomoco.csv"))

### Next step
Once you're happy with the grouping, tell me and I'll point `05_TFActivityRegression_TFScreen.r`
at **`MotifMatrixTFGroups_hocomoco.csv`** (instead of the raw `MotifMatrix_hocomoco.csv`) so the
per-condition regression is run on the TF groups.